In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 6


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 1.4000700786709785
Epoch 2/100, Loss: 1.4122405387461185
Epoch 3/100, Loss: 1.3905235901474953
Epoch 4/100, Loss: 1.3950827568769455
Epoch 5/100, Loss: 1.2326299510896206
Epoch 6/100, Loss: 1.4894364848732948
Epoch 7/100, Loss: 1.289279442280531
Epoch 8/100, Loss: 1.5154844596982002
Epoch 9/100, Loss: 1.2628880441188812
Epoch 10/100, Loss: 1.3203544467687607
Epoch 11/100, Loss: 1.4104238450527191
Epoch 12/100, Loss: 1.3319209478795528
Epoch 13/100, Loss: 1.3053989112377167
Epoch 14/100, Loss: 1.407683502882719
Epoch 15/100, Loss: 1.365093819797039
Epoch 16/100, Loss: 1.337565552443266


Epoch 17/100, Loss: 1.6887242309749126
Epoch 18/100, Loss: 1.2880054339766502
Epoch 19/100, Loss: 1.4856287650763988
Epoch 20/100, Loss: 1.3348507769405842
Epoch 21/100, Loss: 1.4191299080848694
Epoch 22/100, Loss: 1.487378068268299
Epoch 23/100, Loss: 1.3423905037343502
Epoch 24/100, Loss: 1.4582550004124641
Epoch 25/100, Loss: 1.4147736132144928
Epoch 26/100, Loss: 1.300187662243843
Epoch 27/100, Loss: 1.4057817868888378
Epoch 28/100, Loss: 1.4002045914530754
Epoch 29/100, Loss: 1.4700747169554234
Epoch 30/100, Loss: 1.3800175450742245
Epoch 31/100, Loss: 1.3599779680371284
Epoch 32/100, Loss: 1.5599861815571785
Epoch 33/100, Loss: 1.3522536642849445


Epoch 34/100, Loss: 1.2589447125792503
Epoch 35/100, Loss: 1.3883154392242432
Epoch 36/100, Loss: 1.2423914819955826
Epoch 37/100, Loss: 1.4485097602009773
Epoch 38/100, Loss: 1.2880879454314709
Epoch 39/100, Loss: 1.4177812710404396
Epoch 40/100, Loss: 1.3368647806346416
Epoch 41/100, Loss: 1.464466132223606
Epoch 42/100, Loss: 1.4659354165196419
Epoch 43/100, Loss: 1.3627199828624725
Epoch 44/100, Loss: 1.3525348864495754
Epoch 45/100, Loss: 1.3668945245444775
Epoch 46/100, Loss: 1.3457831032574177
Epoch 47/100, Loss: 1.2840848937630653
Epoch 48/100, Loss: 1.2822114154696465
Epoch 49/100, Loss: 1.4170645661652088
Epoch 50/100, Loss: 1.438298862427473


Epoch 51/100, Loss: 1.5186695866286755
Epoch 52/100, Loss: 1.4176189936697483
Epoch 53/100, Loss: 1.434770330786705
Epoch 54/100, Loss: 1.4674342200160027
Epoch 55/100, Loss: 1.518006343394518
Epoch 56/100, Loss: 1.279622919857502
Epoch 57/100, Loss: 1.3170118369162083
Epoch 58/100, Loss: 1.4796951971948147
Epoch 59/100, Loss: 1.5139099173247814
Epoch 60/100, Loss: 1.5026703365147114
Epoch 61/100, Loss: 1.4082512855529785
Epoch 62/100, Loss: 1.4576994515955448
Epoch 63/100, Loss: 1.465553816407919
Epoch 64/100, Loss: 1.4919172786176205
Epoch 65/100, Loss: 1.4594531431794167
Epoch 66/100, Loss: 1.4508749395608902
Epoch 67/100, Loss: 1.3995425701141357


Epoch 68/100, Loss: 1.4571795798838139
Epoch 69/100, Loss: 1.470329910516739
Epoch 70/100, Loss: 1.4017417803406715
Epoch 71/100, Loss: 1.3280435241758823
Epoch 72/100, Loss: 1.3413340784609318
Epoch 73/100, Loss: 1.4034566022455692
Epoch 74/100, Loss: 1.3937992602586746
Epoch 75/100, Loss: 1.3905913531780243
Epoch 76/100, Loss: 1.3378792107105255
Epoch 77/100, Loss: 1.4912447892129421
Epoch 78/100, Loss: 1.4982197545468807
Epoch 79/100, Loss: 1.4392056837677956
Epoch 80/100, Loss: 1.3311544433236122
Epoch 81/100, Loss: 1.4342388585209846
Epoch 82/100, Loss: 1.4107463210821152
Epoch 83/100, Loss: 1.4195230528712273


...

Epoch 32/100, Loss: 2.3400937356054783
Epoch 33/100, Loss: 2.036870487034321
Epoch 34/100, Loss: 2.211562357842922
Epoch 35/100, Loss: 2.3876927495002747
Epoch 36/100, Loss: 2.5258486345410347
Epoch 37/100, Loss: 2.404580947011709
Epoch 38/100, Loss: 2.233687900006771
Epoch 39/100, Loss: 2.351795054972172
Epoch 40/100, Loss: 2.6135549396276474
Epoch 41/100, Loss: 2.2246375754475594
Epoch 42/100, Loss: 2.445409819483757
Epoch 43/100, Loss: 2.4637438878417015
Epoch 44/100, Loss: 2.1500295847654343
Epoch 45/100, Loss: 2.335910454392433
Epoch 46/100, Loss: 2.558194674551487
Epoch 47/100, Loss: 2.376335419714451
Epoch 48/100, Loss: 2.2098120525479317


Epoch 49/100, Loss: 2.174972601234913
Epoch 50/100, Loss: 2.2131516449153423
Epoch 51/100, Loss: 2.2226167917251587
Epoch 52/100, Loss: 2.0799076296389103
Epoch 53/100, Loss: 2.4665712118148804
Epoch 54/100, Loss: 2.1724989637732506
Epoch 55/100, Loss: 2.603059135377407
Epoch 56/100, Loss: 2.458616778254509
Epoch 57/100, Loss: 2.3096629455685616
Epoch 58/100, Loss: 2.3616713881492615
Epoch 59/100, Loss: 2.4119348898530006
Epoch 60/100, Loss: 2.3473062962293625
Epoch 61/100, Loss: 2.315369926393032
Epoch 62/100, Loss: 2.408671498298645
Epoch 63/100, Loss: 2.613143488764763
Epoch 64/100, Loss: 1.994130365550518
Epoch 65/100, Loss: 2.5769308358430862
Epoch 66/100, Loss: 2.230331815779209


Epoch 67/100, Loss: 2.1761606447398663
Epoch 68/100, Loss: 2.301442250609398
Epoch 69/100, Loss: 2.258133575320244
Epoch 70/100, Loss: 2.3123017475008965
Epoch 71/100, Loss: 2.277169607579708
Epoch 72/100, Loss: 2.360477566719055
Epoch 73/100, Loss: 2.4118953570723534
Epoch 74/100, Loss: 2.2916463762521744
Epoch 75/100, Loss: 2.319044142961502
Epoch 76/100, Loss: 2.373064339160919
Epoch 77/100, Loss: 2.419935055077076
Epoch 78/100, Loss: 2.274188496172428
Epoch 79/100, Loss: 2.2847035825252533
Epoch 80/100, Loss: 2.3860160410404205


Epoch 81/100, Loss: 2.290680855512619
Epoch 82/100, Loss: 2.2508343532681465
Epoch 83/100, Loss: 2.120074190199375
Epoch 84/100, Loss: 2.3333514630794525
Epoch 85/100, Loss: 2.2625382021069527
Epoch 86/100, Loss: 2.3843043223023415
Epoch 87/100, Loss: 2.518029600381851
Epoch 88/100, Loss: 2.2166390866041183
Epoch 89/100, Loss: 2.9113278165459633
Epoch 90/100, Loss: 2.2757507786154747
Epoch 91/100, Loss: 2.1614979952573776
Epoch 92/100, Loss: 2.170806936919689
Epoch 93/100, Loss: 2.302172675728798
Epoch 94/100, Loss: 2.1931201219558716
Epoch 95/100, Loss: 2.4784863516688347
Epoch 96/100, Loss: 2.2799884602427483
Epoch 97/100, Loss: 2.0526215322315693


Epoch 98/100, Loss: 2.254490502178669
Epoch 99/100, Loss: 2.1597000621259212
Epoch 100/100, Loss: 2.3372506126761436
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.4243
